# MSDSS：XGBoost 訓練與獨立評估
先上傳 `colab_upload_no_manual_review.zip`。train/test 為 Bagisto＋Shopware，validation 僅 Sylius，Medusa 排除。
目前 small-api-v2 尚未實機採集，會被正式訓練檢查拒絕；不要改旗標繞過。完成採集與證據對帳後重新打包，包含已遮蔽證據。
這份 Notebook 不使用 test 或 Sylius validation 選參數；預先固定 150 棵樹。需要調參時只能在 train 內另作分組驗證。
人工審核不是訓練條件。請上傳新版 colab_upload_no_manual_review.zip；真實 HTTP／原文／證據檢查仍保留。


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, uuid
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("請只上傳一個新版訓練套件 ZIP")
BASE = Path.cwd() / ("msdss-" + uuid.uuid4().hex[:8])
BASE.mkdir()
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    for entry in archive.infolist():
        name = PurePosixPath(entry.filename)
        if name.is_absolute() or '..' in name.parts or not (BASE / entry.filename).resolve().is_relative_to(BASE.resolve()):
            raise ValueError("ZIP 含不合法路徑")
    archive.extractall(BASE)
manifest = json.loads((BASE / 'package-manifest.json').read_text())
for name, expected in manifest.items():
    target = (BASE / name).resolve()
    if not target.is_relative_to(BASE.resolve()) or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise ValueError("套件檔案不一致：" + name)
print("資料和程式位於", BASE)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(BASE / 'model/requirements.txt')])

## 先檢查，通過後再訓練
這一步不訓練、不產生準確率。未送出／未核准、來源不符或跨集合洩漏會列出原因並停止；不要修改旗標繞過。

In [ ]:
check = subprocess.run([sys.executable, '-X', 'utf8', str(BASE / 'model/train.py'),
    '--data-dir', str(BASE / 'data'), '--evidence-root', str(BASE / 'evidence'),
    '--check-only'], capture_output=True, text=True, encoding='utf-8')
print(check.stdout or check.stderr)
if check.returncode != 0:
    raise RuntimeError('資料尚未符合正式訓練條件，請先處理上方問題。')

## 正式訓練
三份 CSV、session、來源證據、原文與來源資料都要通過。未送出候選會停止，不輸出研究分數。
請在看 test／validation 成績之前固定 TREES。CPU 即可執行，不要求 CUDA 或 GPU。

In [ ]:
from datetime import datetime
TREES = 150
OUTPUT = BASE / 'results' / datetime.now().strftime('%Y%m%d-%H%M%S-%f')
subprocess.check_call([sys.executable, '-X', 'utf8', str(BASE / 'model/train.py'),
    '--data-dir', str(BASE / 'data'), '--evidence-root', str(BASE / 'evidence'),
    '--output', str(OUTPUT), '--trees', str(TREES)])

## 分開檢查成績
Accuracy、Precision、Recall、F1、Macro／Weighted／Micro-F1、混淆矩陣、FPR／漏報、ROC-AUC、Average Precision、MCC、Kappa、Log loss、Brier、基準比較與群組區間均另存數據。少於 20 個獨立群組不報區間。
Balanced accuracy 等於各類 Recall 的平均。PR 圖中的 AP 不是梯形積分 PR-AUC。機率未校準。

In [ ]:
from IPython.display import display, Markdown
for split in ('test', 'validation'):
    display(Markdown('## ' + split))
    display(Markdown((OUTPUT / split / 'report.md').read_text()))

In [ ]:
# 可選圖表：CSV/JSON 數據不依賴 Matplotlib。
import csv
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib 未安裝；所有數據仍已保存在 CSV/JSON。')
else:
    for split in ('test', 'validation'):
        folder = OUTPUT / split
        for filename in ('confusion_matrix', 'confusion_matrix_normalized'):
            with (folder / (filename + '.csv')).open(encoding='utf-8-sig', newline='') as handle:
                table = list(csv.reader(handle))
            labels = table[0][1:]
            matrix = [[float(v) for v in row[1:]] for row in table[1:]]
            fig, ax = plt.subplots(figsize=(5, 4))
            image = ax.imshow(matrix, cmap='Blues')
            ax.set(xticks=range(3), yticks=range(3), xticklabels=labels, yticklabels=labels,
                   xlabel='Predicted', ylabel='Actual', title=split + ': ' + filename)
            for i in range(3):
                for j in range(3):
                    ax.text(j, i, f'{matrix[i][j]:.3f}' if filename.endswith('normalized') else f'{matrix[i][j]:.0f}', ha='center', va='center')
            fig.colorbar(image, ax=ax); fig.tight_layout()
            fig.savefig(folder / (filename + '.png'), dpi=160); plt.show()
        with (folder / 'curves.csv').open(encoding='utf-8-sig', newline='') as handle:
            points = list(csv.DictReader(handle))
        for curve in ('ROC', 'PR'):
            fig, ax = plt.subplots(figsize=(5, 4))
            for label in ('normal', 'sqli', 'xss'):
                values = [p for p in points if p['curve'] == curve and p['class'] == label]
                ax.plot([float(p['x']) for p in values], [float(p['y']) for p in values], label=label)
            ax.set(xlabel='FPR' if curve == 'ROC' else 'Recall', ylabel='TPR' if curve == 'ROC' else 'Precision',
                   title=split + ': ' + curve, xlim=(0,1), ylim=(0,1.02))
            ax.legend(); fig.tight_layout(); fig.savefig(folder / (curve + '.png'), dpi=160); plt.show()

## 套件自我檢查與下載
`inference.zip` 內含模型、同版特徵／遮蔽／推論程式、依賴版本與自我檢查樣本。Windows、Ubuntu 各自解壓後都應先執行 `python predict.py --self-test`。Notebook 成功不等於已替你驗證所有作業系統。

In [ ]:
subprocess.check_call([sys.executable, str(OUTPUT / 'inference/predict.py'), '--self-test'])
import shutil
archive = shutil.make_archive(str(BASE / 'results_download'), 'zip', root_dir=OUTPUT)
files.download(archive)